# Class 09 — First decoder: CSP+LDA, one subject

Building the first real motor-imagery decoder: CSP (Common Spatial Patterns) for spatial filtering, feeding into LDA for classification. Evaluated with `GroupKFold` on `session`, so train and test always come from different days — never mixed, to avoid session-level leakage.

## Setup

In [1]:
import numpy as np
import pandas as pd
import torch
import mne
import matplotlib.pyplot as plt

from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score, GroupKFold

from mne.decoding import CSP

from pyriemann.estimation import Covariances
from pyriemann.classification import MDM, TSClassifier

mne.set_log_level("WARNING")
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)


/Users/vicky/Documents/motor_imagery_decoding/motor-imagery-decoding/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load subject 1

In [2]:
dataset = BNCI2014_001()
dataset.subject_list = [1]

paradigm = MotorImagery(n_classes=4, fmin=8, fmax=30, tmin=0.5, tmax=3.5)
X, y, metadata = paradigm.get_data(dataset=dataset, subjects=[1])


Choosing from all possible events


## CSP + LDA pipeline

CSP and LDA are chained inside a single `Pipeline` so that, in every cross-validation fold, CSP is refit only on that fold's train data — it never sees the test epochs while learning its spatial filters. Fitting CSP once on all the data before splitting would leak information from test into train.

In [3]:
csp_lda = Pipeline([
    ("CSP", CSP(n_components=6, reg="ledoit_wolf", log=True, norm_trace=False)),
    ("LDA", LinearDiscriminantAnalysis()),
])

groups = metadata["session"]
cv = GroupKFold(n_splits=2)

scores = cross_val_score(csp_lda, X, y, cv=cv, groups=groups)
print(scores, scores.mean())


[0.79513889 0.74305556] 0.7690972222222222


**Result (subject 1):** `[0.795, 0.743]`, mean **0.769** — well above chance (0.25 with 4 classes).